In [3]:
import json
from tqdm import tqdm
from pathlib import Path

import copy
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [4]:
# Root Path
ROOT = Path("Amazon_products")

# Train and Test Dataset
TRAIN_CORPUS_PATH = ROOT / "train" /  "train_corpus.txt"
TEST_CORPUS_PATH = ROOT / "test" / "test_corpus.txt"

# Taxonomy
CLASSES_PATH = ROOT / "classes.txt"
HIERARCHY_PATH = ROOT / "class_hierarchy.txt"
KEYWORDS_PATH = ROOT / "class_related_keywords.txt"

In [5]:
import matplotlib.pyplot as plt
from collections import defaultdict
import itertools

# ------------------------
# Function for loads
# ------------------------

def load_lines(p: Path):
    with p.open("r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

def load_pid2text(p: Path):
    """TSV: pid \\t text  -> dict[pid]=text"""
    pid2text = {}
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t", 1)
            if len(parts) == 2:
                pid, text = parts
                pid2text[pid] = text
    return pid2text

def load_classes_int(p: Path):
    class_dict = {}
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            label_int, label_str = line.rstrip("\n").split("\t")
            class_dict[int(label_int)] = label_str
    return class_dict

def load_keywords(p: Path):
    keywords = {}
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            key, items = line.rstrip("\n").split(":")
            item_list = [item for item in items.split(",")]
            keywords[key] = item_list
    return keywords

def load_class_graph(p: Path):
    edges = []
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            p, c = map(int, line.rstrip("\n").split("\t"))
            edges.append((p, c))
    return edges

def load_json(path):
    """Load JSON file into Python object."""
    with open(path) as f:
        return json.load(f)

# ------------------------
# Visualization
# ------------------------

def plot_results(results_dict, split="valid", metric="Loss"):
    """
    Plot metric (e.g., loss) values over epochs for multiple models.

    Args:
        results_dict: dict of dicts
            Example:
                results_dict["valid"]["mlp_partial"] = [0.69, 0.65, ...]
        split: "train" | "valid" | "test"
        metric: name of the metric to display (default: Loss)
    """
    assert split in results_dict, f"{split} not in results_dict"

    plt.figure(figsize=(8, 5))

    for label, value_list in results_dict[split].items():
        plt.plot(
            range(1, len(value_list) + 1),
            value_list,
            marker="o",
            label=label
        )

    plt.title(f"{split.capitalize()} {metric} over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()



In [6]:
# ---------- Read-only loads ----------

train_pid2text    = load_pid2text(TRAIN_CORPUS_PATH)
test_pid2text     = load_pid2text(TEST_CORPUS_PATH)
pid2class         = load_classes_int(CLASSES_PATH)
rel_keywords      = load_keywords(KEYWORDS_PATH)
class_graph_edges = load_class_graph(HIERARCHY_PATH)

print(f"#train={len(train_pid2text):,}  #test={len(test_pid2text):,}")

#train=29,487  #test=19,658


# Embeddings using LLM   
OpenAI gpt-4o-mini used

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

CLIENT_KEY = os.environ.get("CLIENT_KEY")

In [10]:
from openai import OpenAI

client = OpenAI(
  api_key=CLIENT_KEY
)

model = "gpt-4o-mini"
product = pid2class[0]
prompt = f"""
You are given a product category name and its related keywords.
Write a neutral and factual definition of this category in one sentence.

Category name: {product}
Related keywords: {", ".join(rel_keywords[product])}

The sentence should explain what this category refers to, 
without using marketing language or mentioning any specific brand or platform.
"""

print(prompt)


You are given a product category name and its related keywords.
Write a neutral and factual definition of this category in one sentence.

Category name: grocery_gourmet_food
Related keywords: snacks, condiments, beverages, specialty_foods, spices, cooking_oils, baking_ingredients, gourmet_chocolates, artisanal_cheeses, organic_foods

The sentence should explain what this category refers to, 
without using marketing language or mentioning any specific brand or platform.



In [11]:
response = client.responses.create(
  model=model,
  input=prompt,
  store=True,
)

print(response.output_text)

Grocery gourmet food refers to a category of high-quality food items that includes snacks, condiments, beverages, specialty foods, spices, cooking oils, baking ingredients, gourmet chocolates, artisanal cheeses, and organic foods, often characterized by their unique flavors and ingredients.


In [12]:
class_llm_texts = []

for i in range(len(pid2class.values())):
    model = "gpt-4o-mini"
    product = pid2class[i]
    prompt = f"""
    You are given a product category name and its related keywords.
    Write a neutral and factual definition of this category in one sentence.

    Category name: {product}
    Related keywords: {", ".join(rel_keywords[product])}

    The sentence should explain what this category refers to, 
    without using marketing language or mentioning any specific brand or platform.
    """
    
    response = client.responses.create(
    model=model,
    input=prompt,
    store=True,
    )
    
    class_llm_texts.append(response.output_text)

In [18]:
output_path = ROOT / "class_llm_texts.txt"

with open(output_path, "w", encoding="utf-8") as f:
    for i, text in enumerate(class_llm_texts):
        f.write(str(i) + "\t" + text.strip() + "\n")

print(f"Saved to {output_path}")

Saved to Amazon_products\class_llm_texts.txt
